## Introduction to the extended version of DiCE (Diverse Counterfactual Explanations)

[Mothilal et al. (2020)](https://dl.acm.org/doi/10.1145/3351095.3372850) introduce their method of generating counterfactual explanations considering _feasibility_, and _diversity_. [Guidotti and Ruggieri (2021)](https://link.springer.com/chapter/10.1007/978-3-030-88942-5_28), claim counterfactual explanations to be robust they should be similar for similar instances when they explain. In this study, in a search to improve the quality and reliability of the counterfactual explanations _robustness_ is found to be helpful and it also introduced in the optimization function.

DiCE-Extended is built upon the [DiCE (Diverse Counterfactual Explanations)](https://github.com/interpretml/DiCE) [(Mothilal et al. 2020)](https://dl.acm.org/doi/10.1145/3351095.3372850) framework by introducing a robustness term in the optimization function.

## Manipulated Optimization Function

The core enhancement in DiCE-Extended is the manipulated optimization function, designed to balance proximity, diversity, and feasibility of counterfactuals. The function is formulated as:

<a id="equation-1"></a>
\begin{equation}
\tag{1}
C(x) = \underset{c_1, ..., c_k}{\text{arg min}}
\frac{1}{2} \sum_{i=1}^{k} yloss(f(c_i), y) +
\frac{\lambda_1}{k} \sum_{i=1}^{k} dist(c_i, x) -
\lambda_2 \cdot dpp\_diversity(c_1, ..., c_k) -
\frac{\lambda_3}{k} \sum_{i=1}^{k} robustness(c_i, c_i')
\end{equation}

- **Proximity Loss**: The first term that averages the distance between generated counterfactuals and the original input ensure the counterfactuals to be as close as possible to the original input.
- **Diversity Loss**: Diversity of the counterfactual explanations is aquired by determinental point process of which loss is represented by the second term and it ensures that _k_ number of counterfactual explanations are generated.
- **Robustness Loss**: [Guidotti (2024)](https://link.springer.com/article/10.1007/s10618-022-00831-6) defines robustness as necessity of similar instances being explained by similar counterfactual explanations such that if $b(x_1)=b(x_2)=y$ then an explainer $f$ should generate counterfactuals $c_1$ and $c_2$ that are similar and can explain $x_1$ and $x_2$. The robustness term that is based on [Dice-Sørensen Coefficient](https://en.wikipedia.org/wiki/Dice-S%C3%B8rensen_coefficient), is adopted from [Bonasera and Carrizosa (2024)](
https://doi.org/10.48550/arXiv.2407.00843).

\begin{equation}
\tag{2}
Robustness(c_i, c_i') = \frac{2 * \lvert c_i \cap c_i' \rvert}{\lvert c_i \rvert + \lvert c_i' \rvert}
\end{equation}


By adjusting the weights $\lambda_1$, $\lambda_2$, $\lambda_3$ counterfactual explanations can be customised by specific needs.

## Metrics and Sensitivity Analysis for Dice Extended


### 1. Robustness Metrics

#### Dice-Sørensen Coefficient

To evaluate robustness, the Dice-Sørensen coefficient measures the similarity between counterfactuals c1 and
c2 generated for similar input instances x1 and x2:

\begin{equation}
\tag{3}
Robustness(c_1, c_2) = \frac{2 * \lvert c_1 \cap c_2 \rvert}{\lvert c_1 \rvert + \lvert c_2 \rvert}
\end{equation}

where:
- $ c_1 $ and $ c_2 $ are binary vectors,
- $ \lvert c_1 \cap c_2 \rvert $: The number of shared (overlapping) features between c1 and c2,
- $ \lvert c_1 \rvert $ and $ \lvert c_2 \rvert $: The total number of features in each counterfactual.

#### Input Perturbation and Stability

Stability under input perturbation measures the solution variance when slight perturbations are introduced
to the input instance. The procedure includes the following steps:

1) **Apply Gaussian Noise:** Perturb the input $x$ by adding Gaussian noise $\delta$ to create perturbed inputs
$x'$:

\begin{equation}
\tag{4}
x' = x + \delta, \quad \delta \sim \mathcal{N}(0, \sigma^2)
\end{equation}

where $\sigma$ is the standard deviation of the noise (e.g., $\sigma = 0.01$).

2) **Generate Counterfactuals:** Generate counterfactual explanations $c_i$ for the original input $x$ and $c_i'$ for the perturbed input $x'$.

3) **Measure Stability:** Compare counterfactuals using a distance metric, such as the Euclidean distance:

\begin{equation}
\tag{5}
Stability = \frac{1}{n} \sum_{i=1}^{n} dist(c_i, c_i')
\end{equation}

where:

\begin{equation}
\tag{6}
dist(c_i, c_i') = \sqrt{\sum_{j=1}^{d} (c_{ij} - c_{ij}')^2}
\end{equation}

$n$ is the total number of input instances, $c_i$ is the counterfactual for the original input, and $c_i'$ is the counterfactual for the perturbed input.

### 2. Counterfactual Quality Measures

#### Fidelity

Fidelity measures how often generated counterfactuals successfully change the model’s prediction:

\begin{equation}
\tag{7}
Fidelity = \frac{\sum_{i=1}^{n} \mathbf{1}(f(c_i) = y_{desired})}{n}
\end{equation}

where:

- $f$: Prediction model,
- $c_i$: Counterfactual instance,
- $y_{desired}$: Target output class,
- $n$: Total number of counterfactuals.

#### Proximity

Proximity measures the average distance between counterfactuals $c_i$ and the original inputs $x_i$:

\begin{equation}
\tag{8}
Proximity = \frac{1}{n} \sum_{i=1}^{n} dist(x_i, c_i)
\end{equation}

The Manhattan distance can be used for simplicity:

\begin{equation}
\tag{9}
dist(x_i, c_i) = \sum_{j=1}^{d} \lvert x_{ij} - c_{ij} \rvert
\end{equation}

#### Diversity

Diversity measures how dissimilar the counterfactuals $c_1, c_2, c_3,\ldots,c_k$ are among themselves:

\begin{equation}
\tag{10}
Diversity = \frac{1}{k(k-1)}\sum_{i_1}^{k}\sum_{j \neq i}^{} dist(c_i, c_j)
\end{equation}

where $k$ is the number of counterfactuals.

### 3. Sensitivity Analysis

#### Objective Function with Weights

The modified loss function in DiCE-Extended is defined as in the [equation 1](#equation-1) where:

- $yloss(f(c_i), y)$: Prediction loss for counterfactual instance $c_i$ relative to the desired outcome $y$,
- $dist(c_i, x)$: Distance metric (e.g., Euclidean or Manhattan) between the counterfactual c_i and the original input $x$,
- $dpp\_diversity(c_1,\ldots,c_k)$: Diversity loss term based on Determinantal Point Process (DPP),
- $Robustness(c_i,c_i')$: Robustness loss measuring similarity of counterfactuals under perturbations.

  The weights $\lambda_1, \lambda_2, \lambda_3$ control the balance between proximity, diversity, and robustness, respectively.

#### Sensitivity Analysis

To perform sensitivity analysis:

1) Vary the weights $\lambda_1, \lambda_2, \lambda_3$ systematically while ensuring:

\begin{equation}
\tag{11}
\lambda_1 + \lambda_2 + \lambda_3 = 1 (for normalization).
\end{equation}

2) Track the changes in the following metrics:

\begin{equation}
\tag{12}
P(\lambda_1, \lambda_2, \lambda_3) = Proximity,
\end{equation}

\begin{equation}
\tag{13}
D(\lambda_1, \lambda_2, \lambda_3) = Diversity,
\end{equation}

\begin{equation}
\tag{14}
R(\lambda_1, \lambda_2, \lambda_3) = Robustness,
\end{equation}

3) Measure the relationship between these metrics and the weights.



In [1]:
import sys
dice_path = "/Users/volk/Documents/bau24-25/thesis/repos/DiCE-X"
sys.path.insert(0, dice_path)

In [2]:
from dice_ml_x.utils import helpers

In [3]:
%load_ext autoreload
%autoreload 2

In [ ]:
from dice_ml_x.benchmarking import Benchmarking
datasets = ["compas-recidivism", "adult-income", "lending-club", "german-credit"]
backends = ["sklearn", "PYT", "TF2"]
methods =['gaussian']
benchmarking = Benchmarking(datasets=datasets,
                            backends=backends,
                            perturbation_methods=methods)
benchmarking.load_and_train(batch_size=16)

""" At the end of the load and train process a dictionary containing the training and counterfactual
generation history is saved following object can be found in the results object:
"""

Benchmarking:   0%|          | 0/12 [00:00<?, ?it/s]

the dataset is : compas-recidivism, the backend is : sklearn, the method is gaussian


Benchmarking:   8%|▊         | 1/12 [00:10<01:54, 10.40s/it, dataset=compas-recidivism, backend=sklearn, method=gaussian]

the dataset is : compas-recidivism, the backend is : PYT, the method is gaussian


/Users/volk/Documents/bau24-25/thesis/repos/DiCE-X/dice_ml_x/explainer_interfaces/dice_pytorch.py:407: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/BucketizationUtils.h:34.)
  binned_indices = torch.bucketize(col_vals, edges, right=False) - 1
Benchmarking:  17%|█▋        | 2/12 [00:14<01:06,  6.62s/it, dataset=compas-recidivism, backend=PYT, method=gaussian]    

...

Diverse Counterfactuals found! total time taken: 00 min 08 sec


the dataset is : german-credit, the backend is : TF2, the method is gaussian


Benchmarking: 100%|██████████| 12/12 [13:28<00:00, 67.40s/it, dataset=german-credit, backend=TF2, method=gaussian]

Diverse Counterfactuals found! total time taken: 00 min 59 sec


' At the end of the load and train process a dictionary containing the training and counterfactual\ngeneration history is saved following object can be found in the results object:\n'

In [13]:
benchmarking.results

{'compas-recidivism': {'sklearn': {'accuracy': 0.5714285714285714,
   'cfs': {'gaussian':       sex   age  priors_count              race c_charge_degree  twoyearrecid
    0    Male  27.0           0.0  African-American               F             1
    0    Male  28.0           0.0  African-American               F             1
    0    Male  26.0           0.0  African-American               F             1
    0  Female  26.0           0.0         Caucasian               F             1
    0    Male  22.7           0.0  African-American               F             1},
   'input_instance': {'gaussian':        sex   age  priors_count       race c_charge_degree
    4096  Male  27.0           0.0  Caucasian               F},
   'time': {'gaussian': 10.215321063995361},
   'model': Pipeline(steps=[('preprocessor',
                    ColumnTransformer(transformers=[('cat',
                                                     Pipeline(steps=[('onehot',
                                  

In [11]:
import pickle

with open('benchmarking_final_results.pkl', 'wb') as res_file:
    pickle.dump(benchmarking.results, res_file)

## Models' accuracies on all four datasets

In [ ]:
# Accuracy table for each dataset

from IPython.display import display
import pandas as pd

accuracy_data = {
    dataset: {
        model: {
            'accuracy':  round(details['accuracy'], 2)
        }
        for model, details in models.items()
    }
    for dataset, models in benchmarking.results.items()
}

rows = []

for dataset, models in accuracy_data.items():
    for model_name, model_info in models.items():
        row = {
            'Dataset': dataset,
            'Model': model_name,
            'Accuracy': model_info.get('accuracy'),
        }
        rows.append(row)


accuracy_df = pd.DataFrame(rows)
display(accuracy_df)

,Dataset,Model,Accuracy
0,compas-recidivism,sklearn,0.57
1,compas-recidivism,PYT,0.66
2,compas-recidivism,TF2,0.65
3,adult-income,sklearn,0.82
4,adult-income,PYT,0.83
5,adult-income,TF2,0.83
6,lending-club,sklearn,0.82
7,lending-club,PYT,0.83
8,lending-club,TF2,0.83
9,german-credit,sklearn,0.76


## Explainers' counterfactual generation time with different datasets

In [22]:
benchmarking.results['adult-income']['PYT']['time']

{'gaussian': 7.1413209438323975}

In [ ]:
# Counterfactual generation time table
time_spent_data = {
    dataset: {
        model: {
            'time': round(details['time']['gaussian'], 2)
        }
        for model, details in models.items()
    }
    for dataset, models in benchmarking.results.items()
}

print(time_spent_data)

rows = []

for dataset, models in time_spent_data.items():
    for model_name, model_info in models.items():
        row = {
            'Dataset': dataset,
            'Model': model_name,
            'Time (ms)': model_info.get('time'),
        }
        rows.append(row)

time_df = pd.DataFrame(rows)
display(time_df)

,Dataset,Model,Time (ms)
0,compas-recidivism,sklearn,10.22
1,compas-recidivism,PYT,2.69
2,compas-recidivism,TF2,30.95
3,adult-income,sklearn,49.08
4,adult-income,PYT,7.14
5,adult-income,TF2,43.72
6,lending-club,sklearn,110.42
7,lending-club,PYT,17.32
8,lending-club,TF2,200.41
9,german-credit,sklearn,217.62


In [121]:
import pickle

with open("benchmarking_results_ran_w_2.pkl", "wb") as f:
    pickle.dump(benchmarking.results, f)

In [122]:
with open("benchmarking_results_ran_w_2.pkl", "rb") as f:
    b_results = pickle.load(f)

b_results

{'compas-recidivism': {'sklearn': {'accuracy': 0.5714285714285714,
   'cfs': {'gaussian':       sex   age  priors_count              race c_charge_degree  twoyearrecid
    0    Male  27.0           0.0  African-American               F             1
    0    Male  28.0           0.0  African-American               F             1
    0    Male  26.0           0.0  African-American               F             1
    0  Female  26.0           0.0         Caucasian               F             1,
    'random':       sex   age  priors_count              race c_charge_degree  twoyearrecid
    0    Male  27.0           0.0  African-American               F             1
    0  Female  18.0           0.0         Caucasian               F             1
    0    Male  18.0           0.0  African-American               F             1
    0    Male  18.0           0.0  African-American               M             1,
    'spherical':       sex   age  priors_count              race c_charge_degree  